# 2 Baseline: Model Without Context

Purpose:
Establish a “before” snapshot so improvements later are visible, attributable, and defensible.
What we’re doing

We will ask a small set of domain-specific questions without providing any of the customer documents. This simulates what happens when a customer “just tries” a model before connecting their data.

Why we’re doing it

* To create a reference point
* To reveal the gap between fluency and groundedness
* To produce a baseline artifact we can later compare against RAG results

What to look for

* Confident language + incorrect facts
* Vague responses where specificity is required
* Inconsistent terminology across runs

Output

A small baseline dataset:
question → model answer → timestamp → metadata
 saved for later comparison and evaluation.

## 2.1 Setup a Key on MaaS
Using your Red Hat SSO log into the Red Hat Demo Platform MaaS service home page:

https://litellm-prod-frontend.apps.maas.redhatworkshops.io/home

## 2.1.1 Create your .env file

In the root q1labs directory, we need to create a .env file containing your API key and endpoint URL.
In a notebook cell, run the following commands — replacing `YOUR_API_KEY` with the key you copied and `YOUR_ENDPOINT_URL` with the API URL from the key details screen:


In [ ]:
# !echo 'API_KEY=your_key_here' > ../.env
# !echo 'ENDPOINT_BASE=https://litellm-prod.apps.maas.redhatworkshops.io/v1' >> ../.env

Check the that the data is correct

In [4]:
!cat ../.env

API_KEY=sk-UFHcLOTZk_73o6YwVQhSiQ
ENDPOINT_BASE=https://litellm-prod.apps.maas.redhatworkshops.io/v1


## 2.2 Testing the connection and API Key


### Load variables from the .env file

In [5]:
import sys
sys.path.insert(0, "..")
from config import API_KEY as key, ENDPOINT_BASE as endpoint_base


## Validate that configration is correct

In [6]:
print("CONFIG DETAILS")
print("=====================")
print(f"API Key = {key}")
print(f"EndPoint Base URL = {endpoint_base}")


CONFIG DETAILS
API Key = sk-UFHcLOTZk_73o6YwVQhSiQ
EndPoint Base URL = https://litellm-prod.apps.maas.redhatworkshops.io/v1


In [7]:
import requests
url_models = f"{endpoint_base}/models"
url_chat = f"{endpoint_base}/chat/completions"

## Test the Connection to MaaS Server`

The following code will test the connection to the MaaS endpoint and ensure you can communicate with the LLMs hosted there.

With the connection working, let’s move on to see what models are available.

In [8]:
headers = {
    "Authorization": f"Bearer {key}",
    "Content-Type": "application/json"
}
data = {
    "model": "granite-3-2-8b-instruct",
    "messages": [{"role": "user", "content": "Hello, world!"}]
}

response = requests.post(url_chat, headers=headers, json=data)
print(response.json())

{'id': 'chatcmpl-f6b5b755856846ff876d69dc6a4a7d37', 'created': 1771341078, 'model': 'granite-3-2-8b-instruct', 'object': 'chat.completion', 'system_fingerprint': None, 'choices': [{'finish_reason': 'stop', 'index': 0, 'message': {'content': 'Hello there! How can I assist you today?', 'role': 'assistant', 'tool_calls': None, 'function_call': None}, 'provider_specific_fields': {'stop_reason': None}}], 'usage': {'completion_tokens': 11, 'prompt_tokens': 12, 'total_tokens': 23, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'service_tier': None, 'prompt_logprobs': None, 'kv_transfer_params': None}


## 2.3 Listing the Models Available to the API Key

In [9]:
headers = {"Authorization": f"Bearer {key}"}

response = requests.get(url_models, headers=headers)
models = response.json()

print("Models available")

# Loop through and print just the names
for model in models['data']:
    print(f"\t- {model['id']}")

Models available
	- granite-4-0-h-tiny
	- microsoft-phi-4
	- qwen3-14b
	- granite-3-2-8b-instruct


## 2.4 Testing the Models’ Existing Knowledge


Ask a common RPG question


Next, we will test the language models’ innate knowledge of our name space. This is what many customers would do. Run the following code in a new cell to define the `test_all_models` function.


Now, run the following code to see what the models know about the Basic Fantasy game


In [11]:
def test_all_models(api_key: str, base_url: str, question: str, max_tokens: int = 1000):

    url_models = f"{base_url}/models"
    url_chat = f"{base_url}/chat/completions"

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    # 1. Get all models
    try:
        response = requests.get(url_models, headers=headers)
        response.raise_for_status()
        models_data = response.json().get("data", [])
    except Exception as e:
        print(f"❌ Failed to fetch models: {e}")
        return

    print(f"\n--- Running prompt against {len(models_data)} models ---\n")

    # 2. Loop through each model
    for model in models_data:
        model_id = model["id"]
        print(f"🤖 Testing Model: {model_id}...")

        payload = {
            "model": model_id,
            "messages": [{"role": "user", "content": question}],
            "max_tokens": max_tokens
        }

        try:
            res = requests.post(url_chat, headers=headers, json=payload)
            res.raise_for_status()

            answer = res.json()["choices"][0]["message"]["content"]
            print(f"✅ Response:\n{answer}\n")

        except Exception as e:
            print(f"❌ Failed to get response from {model_id}: {e}\n")

    print("--- All tests complete ---")

Now, run the following code to see what the models know about the Basic Fantasy game


In [13]:
test_all_models(
    api_key=key,
    base_url=endpoint_base,
    question="What happens if a Thief fails an Open Locks attempt?"
)


--- Running prompt against 4 models ---

🤖 Testing Model: qwen3-14b...
✅ Response:


In **Dungeons & Dragons 5th Edition**, if a Thief (or any character with the **Thieves' Tools** and proficiency in **Sleight of Hand** or **Investigation**) **fails an Open Locks attempt**, the immediate mechanical result is that the lock **remains unopened**. However, the consequences depend on the **specific circumstances** and the **Dungeon Master's (DM) discretion**, as the rules provide some flexibility. Here's a breakdown:

---

### **1. Lock Jams or Becomes Harder to Pick**
- **Standard Outcome**: If the character fails the check, the **lock might jam**, making future attempts more difficult. The DM may **increase the DC** of the lock by **1d4 + 1** (or another amount) for subsequent attempts, depending on the lock's complexity.
- **Example**: A failed attempt on a DC 15 lock might raise its DC to **17** or **18** for future tries.

---

### **2. Triggering Traps or Alarm Systems**
- If the loc

You’ll notice that the output will vary slightly across each model, but the LLM will hedge their bets by saying something to the effect of “In many fantasy role-playing games, including Dungeons & Dragons,”  The customer’s game is not Dungeons & Dragons and we want specific to this game.

Given that the models’ innate answer mentions the customer’s top competitor is certainly a sore point. 

Once again, the models offer results that generalize based on whatever public documents it had access to during training. 

For reference, on page 9 of the rule book, the correct answer is below: 

>Open Locks allows the Thief to unlock a lock without a proper key. It may only be tried once per lock. If the attempt fails, the Thief must wait until they have gained another level of experience before trying again.`

Clearly, we need to augment the model’s response with data specific to the customer’s data and documents. In order to do that, we need to ingest the documents and make them ready for an LLM to consume.
